In [1]:
import sys
print(sys.executable)

/opt/homebrew/Caskroom/miniconda/base/envs/capstone/bin/python


In [2]:
import sys
import os
import requests 

sys.path.insert(0, '/Users/minhee/capstone-project')
os.environ['DJANGO_SETTINGS_MODULE'] = 'capstone_project.settings'

import django
django.setup()

from artifacts.models import Artifact
from sentence_transformers import SentenceTransformer
print("완료!")

/opt/homebrew/Caskroom/miniconda/base/envs/capstone/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


완료!


In [3]:
# 2. 모델 로드
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 49638.21it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def fetch_cma_artifacts(limit=100, skip=0):
    url = "https://openaccess-api.clevelandart.org/api/artworks/"
    params = {
        "has_image": 1,          # 이미지 있는 것만
        "currently_on_view": 1,  # 전시 중인 것만
        "limit": limit,
        "skip": skip,
    }
    response = requests.get(url, params=params)
    return response.json()['data']

In [5]:
# 4. 임베딩용 텍스트 생성
def make_embedding_text(aw):
    culture = aw.get('culture', '')
    if isinstance(culture, list):
        culture = ', '.join(culture)

    parts = [
        aw.get('title', ''),
        aw.get('type', ''),
        aw.get('department', ''),
        aw.get('collection', ''),
        aw.get('technique', ''),
        culture,
        aw.get('description', '') or '',
        aw.get('did_you_know', '') or '',
    ]
    return ' '.join([p for p in parts if p])

In [6]:
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

In [7]:
# 5. DB 저장 + 임베딩 생성
def sync_artifacts(limit=100, skip=0):
    artworks = fetch_cma_artifacts(limit=limit, skip=skip)
    created, updated = 0, 0

    for aw in artworks:
        cleveland_id = aw.get('id')
        if not cleveland_id:
            continue

        # 이미지 URL
        images = aw.get('images') or {}
        image_url = ''
        if images.get('web'):
            image_url = images['web'].get('url', '')

        # culture 처리 (list로 저장)
        culture = aw.get('culture', [])
        if isinstance(culture, str):
            culture = [culture] if culture else []

        # 임베딩 텍스트 + 벡터 생성
        embedding_text = make_embedding_text(aw)
        embedding_vector = model.encode(embedding_text).tolist() if embedding_text else None

        obj, is_created = Artifact.objects.update_or_create(
            cleveland_id=cleveland_id,
            defaults={
                'title': aw.get('title', ''),
                'type': aw.get('type', ''),
                'department': aw.get('department', ''),
                'collection': aw.get('collection', ''),
                'technique': aw.get('technique', ''),
                'culture': culture,
                'creation_date_earliest': aw.get('creation_date_earliest'),
                'creation_date_latest': aw.get('creation_date_latest'),
                'creation_date': aw.get('creation_date', ''),
                'current_location': aw.get('current_location', '') or '',
                'image_url': image_url,
                'description': aw.get('description', '') or '',
                'did_you_know': aw.get('did_you_know', '') or '',
                'accession_number': aw.get('accession_number', ''),
                'tombstone': aw.get('tombstone', '') or '',
                'measurements': aw.get('measurements', '') or '',
                'share_license_status': aw.get('share_license_status', 'CC0'),
                'embedding_text': embedding_text,
                'embedding_vector': embedding_vector,
                'is_active': True,
            }
        )

        if is_created:
            created += 1
        else:
            updated += 1

        if (created + updated) % 10 == 0:
            print(f"진행 중... 생성: {created}, 업데이트: {updated}")

    print(f"완료 - 생성: {created}, 업데이트: {updated}")

sync_artifacts(limit=7000, skip=0)

진행 중... 생성: 0, 업데이트: 10
진행 중... 생성: 0, 업데이트: 20
진행 중... 생성: 0, 업데이트: 30
진행 중... 생성: 0, 업데이트: 40
진행 중... 생성: 0, 업데이트: 50
진행 중... 생성: 0, 업데이트: 60
진행 중... 생성: 0, 업데이트: 70
진행 중... 생성: 0, 업데이트: 80
진행 중... 생성: 0, 업데이트: 90
진행 중... 생성: 0, 업데이트: 100
진행 중... 생성: 0, 업데이트: 110
진행 중... 생성: 0, 업데이트: 120
진행 중... 생성: 0, 업데이트: 130
진행 중... 생성: 0, 업데이트: 140
진행 중... 생성: 0, 업데이트: 150
진행 중... 생성: 0, 업데이트: 160
진행 중... 생성: 0, 업데이트: 170
진행 중... 생성: 0, 업데이트: 180
진행 중... 생성: 0, 업데이트: 190
진행 중... 생성: 0, 업데이트: 200
진행 중... 생성: 0, 업데이트: 210
진행 중... 생성: 0, 업데이트: 220
진행 중... 생성: 0, 업데이트: 230
진행 중... 생성: 0, 업데이트: 240
진행 중... 생성: 0, 업데이트: 250
진행 중... 생성: 0, 업데이트: 260
진행 중... 생성: 0, 업데이트: 270
진행 중... 생성: 0, 업데이트: 280
진행 중... 생성: 0, 업데이트: 290
진행 중... 생성: 0, 업데이트: 300
진행 중... 생성: 0, 업데이트: 310
진행 중... 생성: 0, 업데이트: 320
진행 중... 생성: 0, 업데이트: 330
진행 중... 생성: 0, 업데이트: 340
진행 중... 생성: 0, 업데이트: 350
진행 중... 생성: 0, 업데이트: 360
진행 중... 생성: 0, 업데이트: 370
진행 중... 생성: 0, 업데이트: 380
진행 중... 생성: 0, 업데이트: 390
진행 중... 생성: 0, 업데이트: 400
진행 중... 생

In [8]:
# 6. 확인
total = Artifact.objects.filter(embedding_vector__isnull=False, is_active=True).count()
print(f"임베딩 완료된 유물 수: {total}")

# 샘플 확인
sample = Artifact.objects.filter(embedding_vector__isnull=False).first()
print(f"샘플 유물: {sample.title}")
print(f"임베딩 벡터 길이: {len(sample.embedding_vector)}")

임베딩 완료된 유물 수: 33522
샘플 유물: Leda and the Swan
임베딩 벡터 길이: 384


In [ ]:
import re

# 엑셀에서 허용 안 되는 특수문자 제거
ILLEGAL_CHARACTERS_RE = re.compile(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]')

def clean_text(val):
    if isinstance(val, str):
        return ILLEGAL_CHARACTERS_RE.sub('', val)
    return val

df = df.applymap(clean_text)
df['culture'] = df['culture'].apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))
df.drop(columns=['embedding_vector'], inplace=True)

df.to_excel('/Users/minhee/Desktop/artifacts_db.xlsx', index=False)
print(f"완료! {len(df)}개 유물 저장됨")

In [10]:
from artifacts.models import Artifact
import pandas as pd

df = pd.DataFrame(list(Artifact.objects.all().values()))
df['culture'] = df['culture'].apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))
df.drop(columns=['embedding_vector'], inplace=True)  # 벡터는 너무 길어서 제외

df.to_excel('/Users/minhee/Desktop/artifacts_db.xlsx', index=False)
print(f"완료! {len(df)}개 유물 저장됨")

IllegalCharacterError: Each item in this set has a delicate low-relief design of flowering plum branches over scattered, intersecting lines meant to resemble the cracked-ice surface of a frozen body of water and is signed on the base in gold pigment. While Yohei II produced many fine works in underglaze blue, like those produced by Kiyomizu Shichibei, he also made works in quite different styles later in his career, from the early 1870s. In 1873, he was appointed purveyor to the Industrial Center of Kyoto Prefecture, a designation associated with Kyoto’s efforts to reach an international market through the port of Kobe; and from 1875 until his death, he was involved in national-level projects to present Japanese ceramics across the world. It was during this period that Yohei III was apprenticed to Yohei II, and it has been suggested that this set may in fact be an early example of Yohei III’s work, which he signed with his teacher’s name.Decorated with flowers resembling 